# Smart MCQ Solver - High Score V3 + College Milestones

This notebook solves the MCQ ranking task and also covers the required college milestones.

Main high-score idea:

- Clean question prompts aggressively.
- Match every test question to the same question in train.
- Retrieve the correct answer text from train.
- Map the retrieved answer text back to the test option label.
- Rank top 3 answers for MAP@3.

This is still written as a simple, understandable pipeline.

## Milestone 0 - Orientation & Setup

Accounts/tools expected: Kaggle, GitHub, W&B.  
This notebook runs on Kaggle using the uploaded competition files.

In [11]:
import os
import re
import gc
import random
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from difflib import SequenceMatcher

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

CHOICES = ["A", "B", "C", "D", "E"]

## Load Data

The code below works both on Kaggle and locally. If paths are different, update `DATA_DIR`.

In [12]:
# Kaggle path style
DATA_DIR = "/kaggle/input"

# Auto-find files on Kaggle
train_path = None
test_path = None
sample_path = None

for root, dirs, files in os.walk(DATA_DIR):
    for file in files:
        low = file.lower()
        full = os.path.join(root, file)
        if low == "train.csv":
            train_path = full
        elif low == "test.csv":
            test_path = full
        elif low == "sample_submission.csv":
            sample_path = full

# Local fallback
if train_path is None:
    train_path = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
if test_path is None:
    test_path = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
if sample_path is None:
    sample_path = "/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_path)

print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("Sample:", sample_submission.shape)
train_df.head()

Train: (2000, 8)
Test: (500, 7)
Sample: (500, 2)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


## Milestone 1 - Text Cleaning, TF-IDF, Cosine Similarity, MAP@3

This section performs cleaning, missing value handling, and implements the MAP@3 metric.

In [13]:
def normalize_question(text):
    """Clean prompt and remove template words used in this competition."""
    text = str(text).lower().strip()

    # Remove leading prompt templates. Some use ':' and some use '?'.
    leading_templates = [
        "pick the best possible answer",
        "select the most accurate option",
        "identify the correct statement",
        "which of the following is correct",
        "choose the correct answer",
        "determine the correct option",
    ]

    for temp in leading_templates:
        text = re.sub(r"^\s*" + re.escape(temp) + r"\s*[:?]\s*", "", text)

    # Sometimes this phrase appears again after another prefix.
    text = re.sub(r"^\s*which of the following is correct\s*[:?]?\s*", "", text)

    # Remove ending prompt templates.
    ending_templates = [
        "among the listed options",
        "from the following choices",
        "carefully",
        "based on the given context",
    ]

    for temp in ending_templates:
        text = re.sub(r"\s*" + re.escape(temp) + r"\.?\s*$", "", text)

    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_option(text):
    """Clean option text for exact/fuzzy matching."""
    text = str(text).lower().strip()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def map_at_3(y_true, y_pred_top3):
    scores = []
    for true_label, preds in zip(y_true, y_pred_top3):
        if true_label in preds[:3]:
            scores.append(1.0 / (preds.index(true_label) + 1))
        else:
            scores.append(0.0)
    return float(np.mean(scores))


# Missing value handling
for col in ["prompt"] + CHOICES:
    train_df[col] = train_df[col].fillna("").astype(str)
    test_df[col] = test_df[col].fillna("").astype(str)

train_df["answer"] = train_df["answer"].astype(str).str.strip()

train_df["clean_question"] = train_df["prompt"].apply(normalize_question)
test_df["clean_question"] = test_df["prompt"].apply(normalize_question)

for col in CHOICES:
    train_df[f"clean_{col}"] = train_df[col].apply(normalize_option)
    test_df[f"clean_{col}"] = test_df[col].apply(normalize_option)

print("Unique train questions:", train_df["clean_question"].nunique())
print("Unique test questions:", test_df["clean_question"].nunique())
print("Exact test question overlap with train:", test_df["clean_question"].isin(set(train_df["clean_question"])).sum(), "/", len(test_df))

Unique train questions: 252
Unique test questions: 228
Exact test question overlap with train: 500 / 500


### Baseline: TF-IDF Cosine Similarity

This is included for milestone understanding. It is not the final high-score method.

In [14]:
# Simple TF-IDF similarity between prompt and each option
baseline_predictions = []

for _, row in test_df.iterrows():
    texts = [row["prompt"]] + [row[c] for c in CHOICES]
    tfidf = TfidfVectorizer(stop_words="english").fit_transform(texts)
    sims = cosine_similarity(tfidf[0:1], tfidf[1:]).flatten()
    ranked = [CHOICES[i] for i in np.argsort(-sims)[:3]]
    baseline_predictions.append(ranked)

baseline_preview = pd.DataFrame({
    "ID": test_df["id"],
    "Prediction": [" ".join(x) for x in baseline_predictions]
})
baseline_preview.head()

,ID,Prediction
0,1,A B C
1,2,A B C
2,3,A D C
3,4,A C E
4,5,C A D


## Milestone 3 - RAG Style Retrieval Pipeline

Instead of asking an LLM directly, we retrieve the most relevant training examples for each test question.  
This is a simple RAG-style pipeline: **query → retrieve train context → use retrieved answer text → rank test options**.

In [15]:
def sequence_ratio(a, b):
    return SequenceMatcher(None, a, b).ratio()


def best_option_by_text_similarity(answer_text, test_row):
    """Map a retrieved answer text to A/B/C/D/E in the test row."""
    answer_text = normalize_option(answer_text)

    # 1. Exact normalized match
    for ch in CHOICES:
        if normalize_option(test_row[ch]) == answer_text:
            return ch, 10.0

    # 2. Fuzzy string similarity fallback
    scores = {}
    for ch in CHOICES:
        opt = normalize_option(test_row[ch])
        scores[ch] = sequence_ratio(answer_text, opt)

    best_ch = max(scores, key=scores.get)
    return best_ch, float(scores[best_ch])


# Build dictionary: clean question -> train rows
question_to_train_indices = defaultdict(list)
for idx, q in enumerate(train_df["clean_question"]):
    question_to_train_indices[q].append(idx)

print("Question index built:", len(question_to_train_indices))

Question index built: 252


## Milestone 5 - Ensembling and Top-3 Ranking

The final score combines multiple signals:

1. Exact question lookup from train.
2. Retrieved answer text to test option mapping.
3. Fuzzy answer-option matching for slightly changed options.
4. TF-IDF prompt retrieval fallback.
5. Prior fallback to always return exactly three labels.

In [16]:
# Prompt retrieval fallback for unusual cases
all_questions = pd.concat([train_df["clean_question"], test_df["clean_question"]], axis=0)
prompt_vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words="english", min_df=1)
prompt_vectorizer.fit(all_questions)

train_q_tfidf = prompt_vectorizer.transform(train_df["clean_question"])
test_q_tfidf = prompt_vectorizer.transform(test_df["clean_question"])
prompt_sims = cosine_similarity(test_q_tfidf, train_q_tfidf)


def rank_for_test_row(test_index, top_k_prompt=8):
    row = test_df.iloc[test_index]
    clean_q = row["clean_question"]

    scores = Counter()

    # 1. Strong signal: exact cleaned question match
    exact_indices = question_to_train_indices.get(clean_q, [])

    for idx in exact_indices:
        tr = train_df.iloc[idx]
        train_answer_label = tr["answer"]
        train_answer_text = tr[train_answer_label]

        mapped_label, sim_score = best_option_by_text_similarity(train_answer_text, row)

        # Large weight for exact question retrieval
        scores[mapped_label] += 100.0 + sim_score

    # 2. Fallback/extra signal: nearest prompt retrieval
    nearest_indices = np.argsort(-prompt_sims[test_index])[:top_k_prompt]

    for rank, idx in enumerate(nearest_indices):
        tr = train_df.iloc[idx]
        train_answer_label = tr["answer"]
        train_answer_text = tr[train_answer_label]
        mapped_label, sim_score = best_option_by_text_similarity(train_answer_text, row)

        # Lower weight than exact match
        prompt_weight = float(prompt_sims[test_index, idx])
        scores[mapped_label] += (10.0 * prompt_weight) + sim_score + (1.0 / (rank + 1))

    # 3. If no strong score exists, use prompt-option TF-IDF fallback
    if sum(scores.values()) == 0:
        texts = [row["prompt"]] + [row[c] for c in CHOICES]
        tfidf = TfidfVectorizer(stop_words="english").fit_transform(texts)
        sims = cosine_similarity(tfidf[0:1], tfidf[1:]).flatten()
        for i, ch in enumerate(CHOICES):
            scores[ch] += float(sims[i])

    # 4. Tiny prior to make ordering stable
    prior = {"B": 0.005, "C": 0.004, "A": 0.003, "D": 0.002, "E": 0.001}
    for ch in CHOICES:
        scores[ch] += prior[ch]

    ranked = [ch for ch, score in scores.most_common()]

    # Ensure exactly 3 unique labels
    final = []
    for ch in ranked:
        if ch not in final:
            final.append(ch)
    for ch in CHOICES:
        if ch not in final:
            final.append(ch)

    return final[:3], scores


final_predictions = []
score_debug = []

for i in range(len(test_df)):
    pred, score_obj = rank_for_test_row(i)
    final_predictions.append(pred)
    score_debug.append(score_obj)

submission = pd.DataFrame({
    "ID": test_df["id"],
    "Prediction": [" ".join(x) for x in final_predictions]
})

submission.to_csv("submission.csv", index=False)
submission.head(10)

,ID,Prediction
0,1,A B C
1,2,B C A
2,3,B C A
3,4,E B C
4,5,C B A
5,6,D B C
6,7,E B C
7,8,B C A
8,9,C B A
9,10,B C A


## Diagnostics

This checks whether the retrieval layer is actually being used.  
For this dataset, exact cleaned question overlap should be very high.

In [17]:
exact_overlap = test_df["clean_question"].isin(set(train_df["clean_question"])).sum()
print("Exact cleaned question overlap:", exact_overlap, "/", len(test_df))

# How many rows have a confident exact-answer text mapping?
confident = 0
for i, row in test_df.iterrows():
    inds = question_to_train_indices.get(row["clean_question"], [])
    found = False
    for idx in inds:
        tr = train_df.iloc[idx]
        ans_text = normalize_option(tr[tr["answer"]])
        for ch in CHOICES:
            if normalize_option(row[ch]) == ans_text:
                found = True
    if found:
        confident += 1

print("Rows with exact answer text mapping:", confident, "/", len(test_df))
print("Submission shape:", submission.shape)
print(submission.head())

Exact cleaned question overlap: 500 / 500
Rows with exact answer text mapping: 493 / 500
Submission shape: (500, 2)
   ID Prediction
0   1      A B C
1   2      B C A
2   3      B C A
3   4      E B C
4   5      C B A


## Milestone 2 - Transformer Concepts

Transformer models such as BERT, RoBERTa and DeBERTa create context-aware embeddings using attention.  
For this particular dataset, retrieval is stronger than fine-tuning because test questions heavily overlap with train questions.

The optional code below shows how the MCQ task can be formatted for a transformer, but the final submission above should not be overwritten unless validation proves improvement.

In [18]:
# Optional: keep this section for college milestone explanation.
# We do not run transformer training by default because the retrieval submission is stronger here.

RUN_TRANSFORMER = False

if RUN_TRANSFORMER:
    import torch
    from datasets import Dataset
    from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer

    MODEL_NAME = "microsoft/deberta-v3-base"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    label2id = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
    id2label = {v: k for k, v in label2id.items()}

    train_df["label"] = train_df["answer"].map(label2id)

    def preprocess_mcq(batch):
        first_sentences = []
        second_sentences = []
        for prompt, A, B, C, D, E in zip(batch["prompt"], batch["A"], batch["B"], batch["C"], batch["D"], batch["E"]):
            first_sentences.extend([prompt] * 5)
            second_sentences.extend([A, B, C, D, E])
        tokenized = tokenizer(first_sentences, second_sentences, truncation=True, max_length=256)
        result = {k: [v[i:i+5] for i in range(0, len(v), 5)] for k, v in tokenized.items()}
        result["labels"] = batch["label"]
        return result

    print("Transformer formatting code is ready. Set RUN_TRANSFORMER=True to train.")

## Milestone 4 - MCQ Formulation and Fine-Tuning Notes

For MCQ fine-tuning, each training example is converted into five pairs:

- `(prompt, option A)`
- `(prompt, option B)`
- `(prompt, option C)`
- `(prompt, option D)`
- `(prompt, option E)`

The model learns to score the five choices. LoRA can reduce GPU memory by training only small adapter weights, but this notebook keeps the main final solution simple.

# Final Submission

Submit the generated `submission.csv` file to Kaggle.

In [19]:
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")
submission.head(20)

Saved submission.csv


,ID,Prediction
0,1,A B C
1,2,B C A
2,3,B C A
3,4,E B C
4,5,C B A
5,6,D B C
6,7,E B C
7,8,B C A
8,9,C B A
9,10,B C A
